# SPY · Walk-forward five-session direction experiment
Predict the direction from the 42-candle chart. A compact temporal CNN and a regularized logistic model both read candle and EMA geometry extracted from colored chart pixels. Label 1 means SPY closed higher five trading sessions later.

Select the model, CNN epoch (2, 4 or 8) and DOWN-score quantile (25%, 40% or 55%) by mean annual balanced accuracy on 2022, 2023 and 2024. Each fold trains on earlier years and purges overlapping chart and target dates. Break ties by the worst fold, then earlier epoch and lower quantile. Convert the chosen quantile to a fixed probability threshold using training scores. Fit on the purged training period and freeze all choices before evaluating 2025. Prior experiments have already inspected 2025, so it remains a development set. The 2026 holdout is sealed.

Run all cells on a GPU. The last cell links to a ZIP containing every final training epoch, both models, the frozen decision thresholds, and predictions.

In [ ]:
# @title Read candles, volume and EMA lines from chart pixels
from io import BytesIO
import numpy as np
from PIL import Image

def decode_chart(stream):
    with Image.open(stream) as im:
        if im.size != (1920, 1080):
            raise ValueError(f"Unexpected chart size {im.size}")
        a = np.array(im.convert('RGB'))
    # This release uses one fixed plot template. Index only the two plotting areas.
    centers = np.rint(np.linspace(120, 1722, 42)).astype(int)
    offsets = np.arange(-9, 10)
    panel = a[189:752, centers[:, None]+offsets].astype(np.int16)
    r, g, b = np.moveaxis(panel, -1, 0)
    green = (g-r > 48) & (b-r > 40) & (abs(g-b)<32)
    red = (r-g > 65) & (r-b > 60) & (abs(g-b)<26)
    gold = (r-b>105) & (g-b>55) & (r-g>32)
    blue = (b-r>90) & (b-g>52) & (g-r>25)
    # Centers yield the EMA ordinate. A wider sample survives overlap with a candle.
    def ordinate(mask):
        narrow=mask[:,:,8:11].any(2)
        wide=mask.any(2)
        curve=np.full(42,np.nan)
        for j in range(42):
            ys=np.flatnonzero(narrow[:,j])
            if not len(ys): ys=np.flatnonzero(wide[:,j])
            if len(ys): curve[j]=1-np.median(ys)/562
        known=np.isfinite(curve)
        if known.sum()<32: raise ValueError("EMA could not be recovered from this chart")
        return np.interp(np.arange(42),np.flatnonzero(known),curve[known])
    fast, slow = ordinate(gold), ordinate(blue)
    vol = a[801:966, centers-5].astype(np.int16)
    vr, vg, vb = np.moveaxis(vol,-1,0)
    upvol=(vg-vr>35)&(vb-vr>30)&(abs(vg-vb)<28)
    dnvol=(vr-vg>40)&(vr-vb>40)&(abs(vg-vb)<22)
    is_up = upvol.sum(0)>=dnvol.sum(0)
    height=(upvol|dnvol).sum(0)/165.
    if np.min(height)==0: raise ValueError("Volume unreadable")
    out = []
    for j,up in enumerate(is_up):
        mask = (green if up else red)[:,j,:]
        # The body spans the candle width. The wick is a narrow colored stroke.
        body=np.flatnonzero(mask.sum(1)>=5)
        wick=np.flatnonzero(mask.any(1))
        if not len(wick): # A doji can be fully obscured by an EMA stroke.
            out.append([np.nan,np.nan,np.nan,np.nan])
            continue
        if not len(body): body=np.array([int(np.median(wick))])
        top,bottom=1-body[0]/562,1-body[-1]/562
        o,c=(bottom,top) if up else (top,bottom)
        out.append([o,c,1-wick[0]/562,1-wick[-1]/562])
    candles=np.array(out)
    known=np.isfinite(candles[:,1])
    if known.sum()<39: raise ValueError("Candle colors unreadable")
    for k in range(4):
        candles[:,k]=np.interp(np.arange(42),np.flatnonzero(known),candles[known,k])
    o,c,h,l=candles.T
    # Coordinates are fractions of the current chart's visible high-low range.
    return np.stack([c-fast,c-slow,fast-slow,c-o,h-np.maximum(c,o),
                     np.minimum(c,o)-l,np.diff(c,prepend=c[0]),
                     np.diff(slow,prepend=slow[0]),height],axis=0).astype('float32')


In [ ]:
# @title Verify the data release and set up the two models
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib, json, random, urllib.request, zipfile, warnings
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, log_loss, roc_auc_score

SEED = 42
EPOCHS = 8
CANDIDATE_EPOCHS = (2, 4, 8)
DOWN_QUANTILES = (.25, .40, .55)
FOLD_YEARS = (2022, 2023, 2024)
DATA_SHA = "030d0d8bc9306a80b57be0b66f9f90a0dc41c716514cc1b29964db800158095f"
SOURCE = "0d0a3cfe5acb9410382b5e58a855270620f46078"
RAW = f"https://raw.githubusercontent.com/jamelof23/SPY-EMA-V2/{SOURCE}/data"
URL = "https://github.com/jamelof23/SPY-EMA-V2/releases/download/spy42-v2-images-v1/SPY42_v2_trainval_images.zip"
cache = Path("/content/spy42_v2_data")
cache.mkdir(exist_ok=True, parents=True)
archive = cache / "SPY42_v2_trainval_images.zip"
def digest(path):
    with path.open("rb") as f: return hashlib.file_digest(f, "sha256").hexdigest()
if not archive.exists() or digest(archive)!=DATA_SHA:
    tmp = archive.with_suffix(".walkforward.part")
    urllib.request.urlretrieve(URL,tmp)
    if digest(tmp)!=DATA_SHA: raise ValueError("Image release digest mismatch")
    tmp.replace(archive)
assert torch.cuda.is_available(), "Select a GPU runtime before running the experiment."
device = torch.device("cuda")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

calendar = pd.Index(sorted(pd.read_csv(RAW+"/source/SPY_daily_2019_2025.csv", usecols=["Date"]).Date))
assert calendar.is_unique and calendar[-1]=="2025-12-31"

def load_split(split):
    assert split in ("train", "validation")
    labels = pd.read_csv(f"{RAW}/labels/{split}_labels.csv").sort_values("asof_date").reset_index(drop=True)
    yrs=labels.asof_date.str[:4].astype(int)
    assert ((yrs.between(2020,2024)).all() and len(labels)==1253 if split=="train"
            else (yrs==2025).all() and len(labels)==245)
    assert labels.asof_date.is_unique and set(labels.label_up_5d)=={0,1}
    positions=calendar.get_indexer(labels.asof_date)
    assert np.all(positions>=41)
    assert all(calendar[d+5] < ("2025" if split=="train" else "2026") for d in positions)
    samples=[]
    with zipfile.ZipFile(archive) as z:
        for row in labels.itertuples():
            assert row.image_path==f"images/{split}/SPY_42_asof_{row.asof_date}.png"
            with z.open("spy42_v2/"+row.image_path) as f:
                samples.append(decode_chart(f))
    return np.stack(samples), labels.label_up_5d.to_numpy(), positions

def summarize(x):
    # All entries below come from the same nine pixel-derived channels as the CNN.
    return np.column_stack([x[:,0,-1],x[:,1,-1],x[:,2,-1],x[:,3,-1],
                            x[:,4,-1],x[:,5,-1],x[:,6,-5:].sum(1),
                            x[:,6,-20:].sum(1),x[:,7,-5:].sum(1),
                            x[:,8,-1]-x[:,8,-20:].mean(1)])
class CandleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(nn.Conv1d(9,12,5,padding=2),nn.SiLU(),
                                    nn.Conv1d(12,12,3,padding=1),nn.SiLU())
        self.output=nn.Sequential(nn.Dropout(.20),nn.Linear(36,1))
    def forward(self,x):
        h=self.features(x)
        return self.output(torch.cat([h.mean(2),h[:,:,-5:].mean(2),h[:,:,-1]],dim=1)).flatten()

def fit_cnn(x,y,other=None,epochs=EPOCHS,save_dir=None):
    seed()
    model=CandleCNN().to(device)
    mean=x.mean((0,2),keepdims=True)
    scale=x.std((0,2),keepdims=True).clip(.0001,None)
    def tensor(a): return torch.as_tensor((a-mean)/scale,device=device)
    train=tensor(x)
    target=torch.as_tensor(y,device=device,dtype=torch.float32)
    evaluate=tensor(other) if other is not None else None
    optimizer=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.025)
    priors=[]
    for e in range(1,epochs+1):
        model.train()
        losses=[]
        for batch in torch.randperm(len(y),device=device).split(48):
            logits=model(train[batch])
            loss=nn.functional.binary_cross_entropy_with_logits(logits,target[batch])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(loss.item()*len(batch))
        model.eval()
        with torch.no_grad():
            insample=model(train).sigmoid().cpu().numpy()
            forward=model(evaluate).sigmoid().cpu().numpy() if other is not None else None
        value=sum(losses)/len(y)
        priors.append((insample,forward,value))
        if save_dir:
            torch.save(dict(model=model.state_dict(),mean=torch.from_numpy(mean),scale=torch.from_numpy(scale),
                            epoch=e,seed=SEED,optimizer=optimizer.state_dict()),save_dir/f"epoch_{e:02d}.pt")
            print(f"Final epoch {e:02d} | training BCE {value:.4f} | checkpoint saved", flush=True)
    return priors

def logistic_fit(x,y):
    return make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=400)).fit(summarize(x),y)

def metrics(y,p,t):
    pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(acc=accuracy_score(y,pred),bal=balanced_accuracy_score(y,pred),
                down=tn/(tn+fp),up=tp/(tp+fn),auc=roc_auc_score(y,p),
                down_precision=tn/(tn+fn) if tn+fn else 0.,down_calls=int(tn+fn),
                tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))


In [ ]:
# @title Train on earlier years and lock model choices
import tempfile
out=Path(tempfile.mkdtemp(prefix="spy_ema_walkforward_"))
print("GPU",torch.cuda.get_device_name(0),"| trainable CNN parameters",sum(p.numel() for p in CandleCNN().parameters()),flush=True)
x,y,t=load_split("train")
print("Read",len(x),"training charts. Each chart supplies 42 candle positions and 9 pixel-derived channels.",flush=True)
fold_rows=[]
for year in FOLD_YEARS:
    valid=np.flatnonzero(np.array([d.startswith(str(year)) for d in calendar[t]]))
    # Purge the full validation lookback, plus the five-session training target.
    fit=np.flatnonzero(t+5 < t[valid[0]]-41)
    assert len(fit)>250 and t[fit].max()+5 < t[valid].min()-41
    print(f"Fold {year}: {len(fit)} earlier training dates, {len(valid)} validation dates",flush=True)
    fits=fit_cnn(x[fit],y[fit],x[valid])
    logreg=logistic_fit(x[fit],y[fit])
    prior=logreg.predict_proba(summarize(x[fit]))[:,1]
    after=logreg.predict_proba(summarize(x[valid]))[:,1]
    for kind,e,train_p,valid_p in ([("linear",0,prior,after)] +
            [("cnn",i,ins,vs) for i,(ins,vs,_) in enumerate(fits,1)]):
        for q in DOWN_QUANTILES:
            threshold=float(np.quantile(train_p,q))
            fold_rows.append(dict(model=kind,epoch=e,q=q,year=year,
                train_loss=(fits[e-1][2] if kind=="cnn" else np.nan),threshold=threshold,
                **metrics(y[valid],valid_p,threshold)))
folds=pd.DataFrame(fold_rows)
folds.to_csv(out/"walk_forward_scores.csv",index=False)
candidates=folds[(folds.model=="linear")|folds.epoch.isin(CANDIDATE_EPOCHS)]
ranking=candidates.groupby(["model","epoch","q"],as_index=False).agg(
    mean_bal=("bal","mean"),worst_bal=("bal","min"),mean_acc=("acc","mean"))
ranking=ranking.sort_values(["mean_bal","worst_bal","epoch","q"],ascending=[False,False,True,True])
ranking.to_csv(out/"candidate_ranking.csv",index=False)
selected={r.model:dict(epoch=int(r.epoch),q=float(r.q),mean_bal=r.mean_bal) for r in
          ranking.drop_duplicates("model").itertuples()}
winner=ranking.iloc[0].model
print("Training-period choice:",winner,selected,flush=True)
print("CNN epoch | mean earlier-year balanced accuracy at selected DOWN quantile",flush=True)
for e,rows in folds[(folds.model=="cnn")&(folds.q==selected["cnn"]["q"])].groupby("epoch"):
    print(f"Walk-forward epoch {e:02d} | train BCE {rows.train_loss.mean():.4f} | balanced {rows.bal.mean():.1%} | DOWN {rows.down.mean():.1%} | UP {rows.up.mean():.1%}",flush=True)

first_2025=int(calendar.searchsorted("2025-01-01"))
fit=np.flatnonzero(t+5 < first_2025-41)
print(f"Final training count {len(fit)} after purging the year boundary.",flush=True)
final=fit_cnn(x[fit],y[fit],epochs=selected["cnn"]["epoch"],save_dir=out)
pfit=final[-1][0]
selected["cnn"]["threshold"]=float(np.quantile(pfit,selected["cnn"]["q"]))
linear=logistic_fit(x[fit],y[fit])
selected["linear"]["threshold"]=float(np.quantile(linear.predict_proba(summarize(x[fit]))[:,1],selected["linear"]["q"]))
scaler,reg=linear.steps[0][1],linear.steps[1][1]
(out/"linear_model.json").write_text(json.dumps(dict(mean=scaler.mean_.tolist(),scale=scaler.scale_.tolist(),
        coef=reg.coef_[0].tolist(),intercept=reg.intercept_[0],classes=reg.classes_.tolist())))
lock=dict(winner=winner,models=selected,train_n=len(fit),last_training_asof=calendar[t[fit[-1]]],
          archive_sha=DATA_SHA,target_sessions=5,fold_years=list(FOLD_YEARS),
          minimum_train_to_evaluation_asof_distance=47,seed=SEED)
(out/"locked_choices.json").write_text(json.dumps(lock,indent=2))
print("FROZEN before loading 2025 labels:",json.dumps(lock),flush=True)


In [ ]:
# @title Evaluate 2025 and download the checkpoints
x25,y25,t25=load_split("validation")
assert t[fit[-1]]+5 < t25.min()-41
cnn=CandleCNN().to(device)
state=torch.load(out/f'epoch_{selected["cnn"]["epoch"]:02d}.pt',weights_only=True,map_location=device)
cnn.load_state_dict(state["model"])
cnn.eval()
with torch.no_grad():
    cnn25=cnn((torch.from_numpy(x25).to(device)-state["mean"])/state["scale"]).sigmoid().cpu().numpy()
linear25=linear.predict_proba(summarize(x25))[:,1]
predictions=pd.DataFrame(dict(asof_date=calendar[t25],label_up=y25,cnn=cnn25,linear=linear25))
models=[("cnn",cnn25,selected["cnn"]["threshold"]),("linear",linear25,selected["linear"]["threshold"]),
        ("EMA8 above EMA21",(x25[:,2,-1]>0).astype(float),.5),("Always UP",np.ones(len(y25)),.5)]
result=[]
for name,p,thr in models:
    vals=metrics(y25,p,thr)
    result.append(dict(model=name,threshold=thr,**vals))
    predictions[name+"_direction_up"]=(p>=thr).astype(int)
    print(f"2025 {name} | accuracy {vals['acc']:.1%} | balanced {vals['bal']:.1%} | DOWN {vals['tn']}/{vals['tn']+vals['fp']} | UP {vals['tp']}/{vals['tp']+vals['fn']} | DOWN calls {vals['down_calls']} | DOWN precision {vals['down_precision']:.1%} | AUC {vals['auc']:.3f}",flush=True)
predictions.to_csv(out/"predictions_2025.csv",index=False)
pd.DataFrame(result).to_csv(out/"validation_2025.csv",index=False)
# Circular moving-block bootstrap of the locked winner; conditional on this development year.
rng=np.random.default_rng(SEED)
yp=predictions[winner+"_direction_up"].to_numpy()
sim=[]
n=len(yp)
for _ in range(3000):
    idx=((rng.integers(0,n,size=(n+19)//20)[:,None]+np.arange(20))%n).ravel()[:n]
    truth=y25[idx]; pred=yp[idx]
    if truth.min()==truth.max(): continue
    bal=.5*((pred[truth==0]==0).mean()+(pred[truth==1]==1).mean())
    sim.append([bal,(pred==truth).mean()-(truth==1).mean()])
ci=np.quantile(sim,[.025,.975],axis=0)
print(f"Locked {winner} | approximate 95% interval, balanced accuracy {ci[0,0]:.1%} to {ci[1,0]:.1%}; accuracy difference to always UP {ci[0,1]:+.1%} to {ci[1,1]:+.1%}",flush=True)
(out/"uncertainty.json").write_text(json.dumps(dict(block_sessions=20,replicates=len(sim),balanced_accuracy=ci[:,0].tolist(),accuracy_difference_to_always_up=ci[:,1].tolist())))
bundle=out.parent/(out.name+".zip")
with zipfile.ZipFile(bundle,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()): z.write(f,f.name)
with zipfile.ZipFile(bundle) as z:
    assert z.testzip() is None and len([n for n in z.namelist() if n.startswith("epoch_")])==selected["cnn"]["epoch"]
print("Verified checkpoint bundle",bundle.name,"SHA-256",digest(bundle),"bytes",bundle.stat().st_size,flush=True)
from IPython.display import HTML,display
import base64
encoded=base64.b64encode(bundle.read_bytes()).decode()
display(HTML(f'<p><a download="SPY_walk_forward_checkpoints_and_results.zip" href="data:application/zip;base64,{encoded}">Download all checkpoints, frozen thresholds, per-epoch scores and 2025 predictions</a></p>'))
